In [77]:
import pandas as pd
import numpy as np


In [78]:
df_1 = pd.read_csv('dancers_results_info.csv')
df_2 = pd.read_csv('location_info.csv')
df_3 = pd.read_csv('events_wsdc.csv')


In [79]:
df_1.head()

,dancer_id,event_dance,event_competition,event_role,event_result,event_points,event_name,location_id,event_year,event_month,event_year_and_month
0,1,West Coast Swing,Intermediate,Follower,3,0,Monterey Swing Fest,1,1997,1,1997-01-01
1,1,West Coast Swing,Novice,Follower,4,3,Boogie By The Bay,2,1994,10,1994-10-01
2,1,West Coast Swing,Newcomer,Follower,3,4,Phoenix 4th of July,3,1993,7,1993-07-01
3,2,West Coast Swing,Masters,Follower,F,1,Phoenix 4th of July,3,2003,7,2003-07-01
4,2,West Coast Swing,Masters,Follower,1,10,Phoenix 4th of July,3,1998,7,1998-07-01


In [80]:
df_2.head()

,location_id,event_city,event_state,event_country,latitude,longitude,event_location
0,1,Monterey,California,United States,36.600238,-121.894676,"Monterey, CA"
1,2,San Francisco,California,United States,37.774929,-122.419415,"San Francisco, CA"
2,3,Phoenix,Arizona,United States,33.448377,-112.074037,"Phoenix, AZ"
3,4,Portland,Oregon,United States,45.515232,-122.678385,"Portland, OR"
4,5,Fresno,California,United States,36.737798,-119.787125,"Fresno, CA"


In [81]:
df_3.tail()

,id,name,location,url,date
2380,345,HONEY FEST,"Ufa, Bashkortostan Republic, Russia",https://honeyfest-ufa.ru/,April 2024
2381,346,Swingside Invitational,"Liège, Liège, Belgium",http://www.swingside-invitational.com,October 2024
2382,350,Augsburg Westie Station,"Augsburg, Germany",https://www.augsburgwestiestation.com/home,October 2024
2383,367,Go West Swingfest,"Perth, Western Australia",https://www.gowestdancefest.com/go-west-swingfest,October 2024
2384,381,The Australian Classic West Coast Swing Dance ...,"Gosford, NSW, Australia",http://www.dancewcsa.com.au,January 2025


In [82]:


# Объединяем данные из двух источников (предполагается, что df_1 и df_2 уже загружены)
merged_df = df_1.merge(df_2, on='location_id')

# Убираем лишние пробелы в полях, влияющих на группировку
merged_df['event_country'] = merged_df['event_country'].str.strip()
merged_df['event_state'] = merged_df['event_state'].str.strip()

# Создаем новое поле 'State_final':
# Если event_country равен 'USA' или 'United States', то используем event_state,
# иначе – дублируем event_country.
merged_df['State_final'] = merged_df.apply(
    lambda row: row['event_state'] if row['event_country'] in ['USA', 'United States']
                else row['event_country'],
    axis=1
)

# Группируем данные по event_country, State_final и event_year, считаем уникальные значения event_name и event_year_and_month
events_per_year = merged_df.groupby(
    ['event_country', 'State_final', 'event_year']
)[['event_name', 'event_year_and_month']].nunique().reset_index()

# Создаем сводную таблицу (pivot table)
new_df = events_per_year.pivot_table(
    index=['event_country', 'State_final'],
    columns='event_year',
    values='event_name',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Переименовываем столбцы для ясности: event_country -> Country, State_final -> State
new_df = new_df.rename(columns={'event_country': 'Country', 'State_final': 'State'})

# Сортируем по полю 'State'
new_df = new_df.sort_values(by='State')


In [83]:
new_df

event_year,Country,State,1991,1992,1993,1994,1995,1996,1997,1998,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
27,United States,Alabama,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,1,0
28,United States,Arizona,1,1,1,1,1,1,1,1,...,3,3,3,3,0,2,3,3,3,0
0,Australia,Australia,0,0,0,0,0,0,0,0,...,5,6,6,8,3,2,3,5,5,1
1,Austria,Austria,0,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,1
2,Belgium,Belgium,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,United Kingdom,United Kingdom,0,0,0,0,0,0,0,0,...,4,6,6,8,2,1,2,3,4,1
60,United States,Vermont,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
61,United States,Virginia,0,0,1,1,1,1,1,1,...,2,1,1,1,0,0,1,1,1,0
62,United States,Washington,0,0,0,1,1,0,1,1,...,3,3,3,3,0,0,3,3,2,1


In [87]:
new_df.to_csv('Flourish.csv', index=False)

In [85]:
# Предполагается, что столбцы с годами не являются 'Country' и 'State'
year_columns = [col for col in new_df.columns if col not in ['Country', 'State']]

# Если столбцы с годами представлены в виде строк, можно отсортировать их по числовому значению:
year_columns = sorted(year_columns, key=lambda x: int(x))

# Создаем копию new_df для вычисления кумулятивных значений
new_df_cumulative = new_df.copy()

# Применяем кумулятивную сумму по строкам (axis=1) для столбцов с годами
new_df_cumulative[year_columns] = new_df_cumulative[year_columns].cumsum(axis=1)

# Вывод результата
new_df_cumulative

event_year,Country,State,1991,1992,1993,1994,1995,1996,1997,1998,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
27,United States,Alabama,0,0,0,0,0,0,0,0,...,1,1,1,1,1,2,3,4,5,5
28,United States,Arizona,1,2,3,4,5,6,7,8,...,49,52,55,58,58,60,63,66,69,69
0,Australia,Australia,0,0,0,0,0,0,0,0,...,28,34,40,48,51,53,56,61,66,67
1,Austria,Austria,0,0,0,0,0,0,0,0,...,3,4,5,6,7,7,7,8,9,10
2,Belgium,Belgium,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,United Kingdom,United Kingdom,0,0,0,0,0,0,0,0,...,38,44,50,58,60,61,63,66,70,71
60,United States,Vermont,0,0,0,0,0,0,0,0,...,4,4,4,4,4,4,4,4,4,4
61,United States,Virginia,0,0,1,2,3,4,5,6,...,27,28,29,30,30,30,31,32,33,33
62,United States,Washington,0,0,0,1,2,2,3,4,...,66,69,72,75,75,75,78,81,83,84


In [88]:
new_df_cumulative.to_csv('Flourish_cumulative.csv', index=False)